# Exception Handling in Java
An **exception** is an abnormal condition or event that disrupts the normal flow of program execution.
Exception handling provides a robust way to catch runtime errors and maintain normal application flow.

### Exception Hierarchy Overview:
```
                 Throwable
                 /       \
            Exception     Error (OutOfMemory, StackOverflow - unrecoverable)
            /       \
     Checked         Unchecked (RuntimeException)
   (IOException,     (ArithmeticException, NullPointerException,
    SQLException)     ArrayIndexOutOfBoundsException)
```

### 5 Core Keywords:
- `try`: Encloses risky code that might throw an exception.
- `catch`: Handles the specific thrown exception.
- `finally`: Executes guaranteed clean-up code (runs whether an exception occurred or not).
- `throw`: Explicitly triggers/throws an exception object.
- `throws`: Declares in the method signature that a method may throw specific exceptions.


---
## 1. `try`, `catch`, and `finally` Blocks
- `try` block wraps code that may cause an exception.
- `catch` block catches and handles that specific error.
- `finally` block **always** executes (used for closing resources, streams, etc.).


In [ ]:
try {
    int numerator = 50;
    int denominator = 0; // will cause ArithmeticException
    int result = numerator / denominator;
    System.out.println("Result: " + result);
} catch (ArithmeticException e) {
    System.out.println("[Handled] Cannot divide by zero: " + e.getMessage());
} finally {
    System.out.println("[Finally] This block always executes regardless of an exception.");
}


---
## 2. Multiple Catch Blocks
- A single `try` can have multiple `catch` blocks.
- Specific exceptions must be caught **before** more general exceptions (`Exception`).


In [ ]:
try {
    int[] numbers = {10, 20, 30};
    System.out.println(numbers[5]); // ArrayIndexOutOfBoundsException
} catch (ArithmeticException e) {
    System.out.println("Arithmetic Exception caught: " + e.getMessage());
} catch (ArrayIndexOutOfBoundsException e) {
    System.out.println("Array Index Exception caught: " + e.getMessage());
} catch (Exception e) {
    System.out.println("General Exception caught: " + e.getMessage());
}


---
### 2.1 Java 7+ Multi-Catch Block (Pipe `|` Syntax) (Topic 67)
Instead of writing duplicate catch blocks with identical handling logic, Java 7+ allows catching multiple unrelated exceptions in a **single `catch` block** using the pipe (`|`) operator:
- The catch parameter is implicitly **`final`** (cannot be reassigned).
- The exceptions combined must **not** have a parent-child relationship (e.g. `catch (IOException | FileNotFoundException e)` is illegal because `FileNotFoundException` is a subclass of `IOException`).


In [ ]:
try {
    String input = null;
    // Simulating scenario that can throw either NullPointerException or NumberFormatException
    int parsedNumber = Integer.parseInt(input);
} catch (NullPointerException | NumberFormatException e) {
    // Single concise handler for both exceptions
    System.out.println("[Multi-Catch Handler] Caught either Null or NumberFormat error: " + e.getClass().getSimpleName());
}


---
## 3. `throw` vs `throws`
- **`throw`**: Used inside a method body to explicitly throw an exception (`throw new ExceptionType();`).
- **`throws`**: Used in a method signature to declare that the method might throw exceptions to its caller.


In [ ]:
class Validator {
    // 'throws' in method signature declares the possibility of an exception
    static void validateAge(int age) throws IllegalArgumentException {
        if (age < 18) {
            // 'throw' explicitly throws an exception instance
            throw new IllegalArgumentException("Age must be 18 or above to register.");
        }
        System.out.println("Registration successful! Age: " + age);
    }
}

try {
    Validator.validateAge(16); // causes exception
} catch (IllegalArgumentException e) {
    System.out.println("Caught exception from Validator: " + e.getMessage());
}


---
## 4. Custom (User-Defined) Exception
You can define your own exceptions by extending `Exception` (checked) or `RuntimeException` (unchecked).


In [ ]:
// Custom Checked Exception
class InsufficientFundsException extends Exception {
    public InsufficientFundsException(String message) {
        super(message);
    }
}

class BankAccount {
    private double balance = 1000.0;

    public void withdraw(double amount) throws InsufficientFundsException {
        if (amount > balance) {
            throw new InsufficientFundsException("Withdrawal of Rs. " + amount + " failed. Balance is only Rs. " + balance);
        }
        balance -= amount;
        System.out.println("Withdrawn: Rs. " + amount + ", Remaining: Rs. " + balance);
    }
}

BankAccount account = new BankAccount();
try {
    account.withdraw(1500.0); // Exceeds balance
} catch (InsufficientFundsException e) {
    System.out.println("[Custom Exception Handled] " + e.getMessage());
}


---
### 4.1 User-Defined Unchecked Exception (`extends RuntimeException`) (Topic 68)
- **Checked Exceptions (`extends Exception`)**: Must be either handled via `try-catch` or declared in method signature via `throws` (enforced at compile-time).
- **Unchecked Exceptions (`extends RuntimeException`)**: Runtime logical/input bugs; does **not** force the caller to write `throws` or `try-catch` (cleaner code for programming errors).


In [ ]:
// Custom Unchecked Exception
class InvalidAgeException extends RuntimeException {
    public InvalidAgeException(String message) {
        super(message);
    }
}

class VotingSystem {
    // No 'throws' clause required for unchecked RuntimeException!
    public static void verifyAge(int age) {
        if (age < 18) {
            throw new InvalidAgeException("Voter eligibility denied: Age " + age + " is below 18.");
        }
        System.out.println("Age " + age + " is verified and eligible to vote.");
    }
}

try {
    VotingSystem.verifyAge(15);
} catch (InvalidAgeException e) {
    System.out.println("[Custom Unchecked Exception Handled]: " + e.getMessage());
}


---
## 5. Exception Propagation
When an exception occurs in a method, the JVM searches the call stack backwards for a matching `catch` handler:
- **Unchecked Exceptions (`RuntimeException`)**: Automatically propagate up the call stack until caught or until reaching the `main` method (terminating the thread).
- **Checked Exceptions**: Must be declared using `throws` at every level of the propagation chain unless caught.


In [ ]:
class PropagationDemo {
    static void level3() {
        int result = 100 / 0; // ArithmeticException occurs here
    }

    static void level2() {
        level3(); // propagates from level3 to level2
    }

    static void level1() {
        try {
            level2(); // propagates from level2 to level1
        } catch (ArithmeticException e) {
            System.out.println("[Handled in level1] Exception caught after stack propagation: " + e.getMessage());
        }
    }
}

PropagationDemo.level1();


---
## 6. Garbage Collection in Java
**Garbage Collection (GC)** is the automatic process in the JVM that frees memory occupied by unreferenced/unreachable heap objects.

### How an Object Becomes Eligible for GC:
1. **Nulling a reference**: `obj = null;`
2. **Reassigning a reference**: `obj1 = obj2;` (original `obj1` becomes unreachable)
3. **Island of Isolation**: Two objects reference each other, but neither is referenced from any active stack thread root.

- `System.gc()` / `Runtime.getRuntime().gc()`: Suggests that the JVM run the garbage collector.
- `finalize()`: Historical cleanup callback method on `Object` (deprecated in Java 9+; modern Java uses `AutoCloseable` / `Cleaner`).


In [ ]:
class GcDemo {
    String objName;
    GcDemo(String name) { this.objName = name; }
}

// 1. Nulling a reference
GcDemo g1 = new GcDemo("Target 1");
g1 = null; // Eligible for GC

// 2. Reassigning reference
GcDemo g2 = new GcDemo("Target 2A");
GcDemo g3 = new GcDemo("Target 2B");
g2 = g3; // Original "Target 2A" is now eligible for GC

// Requesting Garbage Collection
System.gc();
System.out.println("Garbage collection requested successfully.");


---
### 6.1 Island of Isolation & `finalize()` Execution (Topics 60 & 61)
- **Island of Isolation**: Occurs when two or more objects reference each other in a cycle, but their root stack pointers are severed. Even though their internal reference count is > 0, they are completely unreachable from any GC Root thread and the JVM reclaims both.
- **`finalize()` Method**: Defined in `java.lang.Object` (`protected void finalize() throws Throwable`). The Garbage Collector historically invoked this method on an object before destroying it to free unmanaged resources. Deprecated in Java 9+ because it is non-deterministic and can delay memory reclamation; modern Java uses `AutoCloseable` / `Cleaner`.


In [ ]:
class IsolationNode {
    String nodeName;
    IsolationNode partner; // reference to another object

    IsolationNode(String name) {
        this.nodeName = name;
    }

    // Overriding finalize (demonstration of lifecycle cleanup hook)
    @Override
    @SuppressWarnings("deprecation")
    protected void finalize() throws Throwable {
        System.out.println("[finalize() invoked by GC Thread] Cleaning up: " + nodeName);
        super.finalize();
    }
}

// 1. Creating Island of Isolation
IsolationNode nodeA = new IsolationNode("Island-Node-A");
IsolationNode nodeB = new IsolationNode("Island-Node-B");

// Form circular link
nodeA.partner = nodeB;
nodeB.partner = nodeA;

// Cut roots from stack!
nodeA = null;
nodeB = null;
// Now nodeA and nodeB form an isolated island in heap. Both are eligible for GC!

System.gc(); // Suggesting JVM to run Garbage Collection
System.out.println("Island of isolation created and GC triggered.");
